# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {getattr(metadata,'cite_as', getattr(metadata,'citeAs', None))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities are referenced by their `@id` fields as per the dataset Croissant schema.

In [ ]:
# List all available record sets by @id
print('Available Record Sets:')
record_sets = getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))
if not record_sets:
    print('No RecordSets found in the metadata. Inspect metadata for available entities.')
else:
    for rset in record_sets:
        print(f"- ID: {rset['@id']}")
        print(f"  Name: {rset.get('name', 'N/A')}")

    # Show fields (columns) for each record set
    print('\nRecord Sets and Fields:')
    for rset in record_sets:
        print(f"- RecordSet @id: {rset['@id']}")
        if 'fields' in rset:
            for field in rset['fields']:
                field_id = field.get('@id', 'N/A')
                print(f"    - Field @id: {field_id}, name: {field.get('name', 'N/A')}")
        elif 'columns' in rset:
            for column in rset['columns']:
                column_id = column.get('@id', 'N/A')
                print(f"    - Column @id: {column_id}, name: {column.get('name', 'N/A')}")
        else:
            print("    No fields or columns found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Replace the list of `record_set_ids` with those you identified above.
Refer to each field or record set exclusively by their `@id` field.

In [ ]:
# Identify record set @ids (example: use real ones from previous output)
# For this dataset, let's programmatically extract them (if any present)
record_sets_metadata = getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))
record_set_ids = [rset['@id'] for rset in record_sets_metadata] if record_sets_metadata else []

dataframes = {}

for record_set_id in record_set_ids:
    # Load all records as a list of dicts
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame loaded for RecordSet {record_set_id}. Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"\nFirst 5 rows of {first_record_set_id}:")
    display(dataframes[first_record_set_id].head())
else:
    print("No RecordSets could be loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes handling outliers, transforming numeric fields, or grouping data by key attributes.

**Note:** Make sure to reference all fields by their `@id`.

In [ ]:
# For demonstration, select a numeric field from the first available record set.
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Attempt to auto-select a float or integer column by inspecting dtype or common numeric names
    numeric_field_id = None
    for col in df.columns:
        if any(key in col.lower() for key in ["count", "mw", "weight", "coverage", "abundance", "id"]):
            try:
                # If convertible to numeric, use it
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field_id = col
                    break
            except Exception:
                continue
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if numeric_field_id is None:
        print("No numeric field found in DataFrame for EDA.")
    else:
        print(f"Using numeric field '@id': {numeric_field_id}")

        # Filter records where the value > threshold (change as needed)
        threshold = df[numeric_field_id].quantile(0.9)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (example: any string/categorical field)
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No record sets with data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if record_set_ids and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize grouped mean if group_field is available
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,4))
        display_df = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
        sns.barplot(y=group_field, x=numeric_field_id, data=display_df)
        plt.title(f"Top 20 {group_field}s by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset containing mass spectrometry analysis of proteins from stimulated human mast cells, inspected the Croissant schema, loaded record sets by their `@id`, explored the available fields, filtered and normalized a numeric variable using `@id` references, and visualized distributions.

Further analysis can be performed by leveraging additional record sets or combining the data as required for your research question.
